<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 05 · Compare & score

Load the four saved result sets, grade each against `test_queries.md`, and build the before/after comparison — the actual deliverable. Verdicts are assistive: `match`/`mismatch` for single-value L1–L3, `trap_ok`/`trap_failed` for the Q12 Golden-Yield trap, and `manual_review` for multi-value questions (Q5, Q8, L4) that need a human/LLM judge.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec
import pandas as pd

config = ec.EvalConfig.from_env()
questions = ec.parse_test_queries()
EXP = list(ec.EXPERIMENTS)

### Per-experiment verdict counts

In [ ]:
graded = {}
for name in EXP:
    try:
        res = ec.load_results(config, name)
    except FileNotFoundError:
        print(f'(skipping {name}: no results file — run its notebook first)')
        continue
    graded[name] = ec.grade_experiment(questions, res)

summary = pd.DataFrame({n: ec.score_summary(g) for n, g in graded.items()}).fillna(0).astype(int)
summary

### Per-question verdict matrix (rows = questions, columns = experiments)

In [ ]:
matrix = {}
for name, g in graded.items():
    matrix[name] = {row['id']: row['verdict'] for row in g}
qorder = [q['id'] for q in questions]
levels = {q['id']: q['level'] for q in questions}
mat = pd.DataFrame(matrix).reindex(qorder)
mat.insert(0, 'level', [levels[i] for i in qorder])
mat

### Side-by-side answers for one question (change the id to inspect)

In [ ]:
qid = 'Q1'
for name in EXP:
    try:
        res = {r['id']: r for r in ec.load_results(config, name)}
    except FileNotFoundError:
        continue
    r = res.get(qid, {})
    print(f'--- {name} ---')
    print('  answer:', str(r.get('answer_summary'))[:160])
    print('  sql   :', str(r.get('sql'))[:200])
    print()

### Optional: bar chart of `match` + `trap_ok` (correct) per experiment

In [ ]:
try:
    import matplotlib.pyplot as plt
    correct = {n: sum(1 for row in g if row['verdict'] in ('match', 'trap_ok'))
               for n, g in graded.items()}
    plt.figure(figsize=(6, 3))
    plt.bar(list(correct.keys()), list(correct.values()))
    plt.ylabel('auto-correct (match + trap_ok)')
    plt.title('Auto-graded correctness by experiment')
    plt.xticks(rotation=15)
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed; skipping chart')

**Reading the results.** The story is the *delta* across columns: jargon/metric questions that are `mismatch`/`trap_failed` under `basic` should move toward `match`/`trap_ok` as structured semantics arrive (`wren_base` → `wren_bi`). `context_dump` shows how far raw context alone gets you. Hand-grade the `manual_review` rows (Q5, Q8, Q13–Q15) using the side-by-side cell above.

## v2 · Exact ground-truth scoring (incl. cross-schema Q16-Q18)

The verdict matrix above uses the *assistive* grader (`manual_review` for
multi-value questions). For an aggregable headline — including the cross-schema-only
Q16-Q18 from the `seagate_multi` fixture — score the saved results with the exact
ground-truth scorer instead. Set `RESULTS_DIR` to the fixture you ran
(`results/` for single-schema, `results/seagate_multi/` for the split fixture).

In [ ]:
import seagate_scoring as scoring

RESULTS_DIR = config.results_dir  # or: pathlib.Path('results/seagate_multi')

def exact_score(name):
    path = RESULTS_DIR / f'{name}.json'
    if not path.exists():
        return None
    import json as _json
    res = _json.loads(path.read_text())
    verdicts = {r['id']: scoring.score_result(
        r['id'], r.get('result_rows', []), r.get('answer_summary')) for r in res}
    correct = sum(1 for v in verdicts.values() if v in scoring.CORRECT_VERDICTS)
    return {'correct': correct, 'total': len(verdicts), 'verdicts': verdicts}

headline = {}
for name in EXP:
    s = exact_score(name)
    if s:
        headline[name] = f"{s['correct']}/{s['total']}"
print('Exact-scored correctness:', headline)

In [ ]:
# Cross-schema-only breakout (Q16-Q18): the direct read on the multi-schema feature.
cross = set(scoring.CROSS_SCHEMA_ONLY)
for name in EXP:
    s = exact_score(name)
    if not s:
        continue
    got = sum(1 for q in cross if s['verdicts'].get(q) in scoring.CORRECT_VERDICTS)
    if any(q in s['verdicts'] for q in cross):
        print(f"{name:>16}: cross-schema-only {got}/3 "
              f"({[s['verdicts'].get(q) for q in sorted(cross)]})")